# 02 – Export Student Embeddings (Local)

Run this notebook **after downloading the student checkpoint from Kaggle**.
The 1D-CNN runs inference over all payload rows and writes `student_embeddings.npy`.

**Prerequisites**
- `payload_256.npy` – from `01_extract_payload_from_pcap.ipynb`
- `student_cnn_best.pt` – checkpoint downloaded from Kaggle notebook
  `../kaggle/01_distill_student_cnn.ipynb` (inside `student_results.zip`)

**Output** → `data/processed/student_embeddings.npy`  ← `(N_packets, D)` float32

**Next step** → `03_build_three_tier_graph.ipynb`

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
PAYLOAD_NPY        = "data/interim/payload_dataset/payload_256.npy"
# Path to the student checkpoint.
STUDENT_CHECKPOINT = "outputs/student_cnn/student_cnn_best.pt"
OUTPUT_PATH        = "data/processed/student_embeddings.npy"

# ── Inference settings ─────────────────────────────────────────────────────────
BATCH_SIZE    = 2048
DROPOUT       = 0.1
L2_NORMALIZE  = True    # keep True for cosine similarity in graph building
FP16_OUTPUT   = False   # set True to halve disk usage (float16 vs float32)
DEVICE        = "auto"  # auto | cpu | cuda | cuda:0

In [4]:
from pathlib import Path

# Tìm project root (thư mục chứa pyproject.toml)
_p = Path.cwd()
while _p != _p.parent:
    if (_p / "pyproject.toml").exists():
        break
    _p = _p.parent
PROJECT_ROOT = _p
print(f"Project root: {PROJECT_ROOT}")

payload_path = PROJECT_ROOT / PAYLOAD_NPY
ckpt_path    = PROJECT_ROOT / STUDENT_CHECKPOINT

assert payload_path.exists(), f"payload_256.npy not found: {payload_path}"
assert ckpt_path.exists(), (
    f"Student checkpoint not found: {ckpt_path}\n"
    "Download student_results.zip from Kaggle and extract student_cnn_best.pt to models/"
)

import numpy as np
arr = np.load(payload_path, mmap_mode="r")
print(f"payload_256.npy  : {arr.shape}  dtype={arr.dtype}")
print(f"student_cnn_best.pt: {ckpt_path.stat().st_size / 1e6:.1f} MB")

Project root: d:\Tai_lieu_nam_ba\Hoc_ky_2\Do_an_chuyen_nganh\Do-an-chuyen-nganh_NT114
payload_256.npy  : (5261944, 256)  dtype=uint8
student_cnn_best.pt: 5.9 MB


In [7]:
import subprocess, sys
from pathlib import Path

(PROJECT_ROOT / OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", "-m",
    "graphslm_ids.offline_path.training.export_student_embeddings",
    "--payload-npy",   PAYLOAD_NPY,
    "--checkpoint",    STUDENT_CHECKPOINT,
    "--output-path",   OUTPUT_PATH,
    "--batch-size",    str(BATCH_SIZE),
    "--dropout",       str(DROPOUT),
    "--device",        DEVICE,
]

if L2_NORMALIZE:
    cmd += ["--l2-normalize"]
else:
    cmd += ["--no-l2-normalize"]

if FP16_OUTPUT:
    cmd += ["--fp16-output"]

print("$", " ".join(cmd))
proc = subprocess.Popen(
    cmd, cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    encoding='utf-8', errors='replace', bufsize=1,
)
_prev_progress = False
for line in proc.stdout:
    text = line.rstrip('\n\r')
    is_progress = '%|' in text
    if is_progress:
        if _prev_progress:
            sys.stdout.write('\r' + text)
        else:
            sys.stdout.write(text)
    else:
        if _prev_progress:
            sys.stdout.write('\n')
        sys.stdout.write(text + '\n')
    sys.stdout.flush()
    _prev_progress = is_progress
if _prev_progress:
    sys.stdout.write('\n')
proc.wait()
if proc.returncode != 0:
    raise subprocess.CalledProcessError(proc.returncode, cmd)

$ d:\v\nt114\Scripts\python.exe -u -m graphslm_ids.offline_path.training.export_student_embeddings --payload-npy data/interim/payload_dataset_14gb/payload_256.npy --checkpoint outputs/student_cnn/student_cnn_best.pt --output-path data/processed/student_embeddings_14gb.npy --batch-size 2048 --dropout 0.1 --device auto --l2-normalize
[cpu] 16 cores → 12 infer threads + 4 data workers
[disk] output size 16.16 GB, free 420.9 GB — OK

Export student embeddings: 100%|██████████| 2570/2570 [10:49<00:00,  3.96it/s]
[OK] Student embeddings saved: data\processed\student_embeddings_14gb.npy
[OK] Metadata saved: data\processed\student_embeddings_14gb.meta.json


In [8]:
# Verify output.
import numpy as np
from pathlib import Path

out_path = PROJECT_ROOT / OUTPUT_PATH
emb = np.load(out_path, mmap_mode="r")
print(f"student_embeddings.npy: {emb.shape}  dtype={emb.dtype}")
print(f"File size             : {out_path.stat().st_size / 1e9:.2f} GB")

# Quick sanity check: L2 norms should be ~1.0 when L2_NORMALIZE=True
sample = emb[:1000].astype('float32')
norms = (sample ** 2).sum(axis=1) ** 0.5
print(f"L2 norms (first 1000 rows): mean={norms.mean():.4f}  std={norms.std():.4f}")

student_embeddings.npy: (5261944, 768)  dtype=float32
File size             : 16.16 GB
L2 norms (first 1000 rows): mean=1.0000  std=0.0000
